# Why we win (or lose) vs full retrain and TTA

Analytical and numerical study: identify *when* regime-aware reuse beats always-retrain or test-time adaptation, and *why*.

Uses three-way benchmark results: `benchmarks/results/baseline_comparison/three_way_results.json`.

## 1. Load results and classify by scenario

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Path: run from repo root (benchmarks/...) or from notebooks/ (../benchmarks/...)
results_path = Path("benchmarks/results/baseline_comparison/three_way_results.json")
if not results_path.exists():
    results_path = Path("../benchmarks/results/baseline_comparison/three_way_results.json")
with open(results_path) as f:
    raw = json.load(f)

# Build table, skip errors
rows = []
for name, r in raw.items():
    if "error" in r:
        continue
    rows.append({
        "dataset": name,
        "baseline_mape": r["baseline_mape"],
        "tta_mape": r["tta_mape"],
        "regime_mape": r["regime_mape"],
        "baseline_time": r["baseline_time"],
        "tta_time": r["tta_time"],
        "regime_time": r["regime_time"],
    })

df = pd.DataFrame(rows)

def scenario_type(name: str) -> str:
    if name.startswith("synth_multi_regime"):
        return "multi_regime"
    if name.startswith("synth_covid_shock"):
        return "covid_shock"
    if name.startswith("synth_recurring"):
        return "recurring"
    if name.startswith("stock_"):
        return "stock"
    if name.startswith("econ_"):
        return "econ"
    return "other"

df["scenario"] = df["dataset"].apply(scenario_type)
df.head(10)

## 2. Define wins: regime vs baseline, regime vs TTA

Use a small tolerance (e.g. 1% MAPE) to call a tie.

In [ ]:
TIE_MAPE = 1.0  # within 1% MAPE = tie

def vs_baseline(row):
    diff = row["baseline_mape"] - row["regime_mape"]
    if diff > TIE_MAPE:
        return "regime_wins"
    if diff < -TIE_MAPE:
        return "baseline_wins"
    return "tie"

def vs_tta(row):
    diff = row["tta_mape"] - row["regime_mape"]
    if diff > TIE_MAPE:
        return "regime_wins"
    if diff < -TIE_MAPE:
        return "tta_wins"
    return "tie"

df["vs_baseline"] = df.apply(vs_baseline, axis=1)
df["vs_tta"] = df.apply(vs_tta, axis=1)

print("Regime vs Baseline:", df["vs_baseline"].value_counts().to_dict())
print("Regime vs TTA:", df["vs_tta"].value_counts().to_dict())

## 3. Where does regime win? By scenario

In [ ]:
print("=== Regime vs Baseline by scenario ===")
print(pd.crosstab(df["scenario"], df["vs_baseline"]))
print()
print("=== Regime vs TTA by scenario ===")
print(pd.crosstab(df["scenario"], df["vs_tta"]))

## 4. Why our accuracy can be *better* than full-data training (core reasoning)

**Puzzle:** How can a model that reuses a checkpoint (trained on a subset of data) beat a model that is retrained on **all** accumulated data? More data should help—so why do we sometimes win?

**Answer: Full-data training fits one model to a *mixture* of regimes. We use a *specialist* for the current regime. On data from that regime, the specialist can have lower loss than the mixture-optimal model.**

### What “full retrain” actually optimizes

- At each step, the baseline fits a single model on **all data so far**: regime 1 + regime 2 + … + current batch.
- So the baseline minimizes loss over the **mixture distribution**  
  \( P_{\text{mix}} = \frac{n_1}{N} P_1 + \frac{n_2}{N} P_2 + \cdots \)  
  where \(P_k\) is the distribution of regime \(k\) and \(n_k\) is the number of points from regime \(k\).
- The resulting model is **optimal for predicting under \(P_{\text{mix}}\)**, not for predicting under the **current** regime’s distribution \(P_{\text{current}}\).

### Why the mixture-optimal model is worse on the current regime

- When the **test batch** (and the next period we care about) is from **one** regime, say regime A, we care about error under \(P_A\).
- The **Bayes-optimal predictor for \(P_A\)** minimizes \(E_{Y \sim P_A}[L(Y, \hat{Y})]\). For MSE, that is the conditional mean \(E[Y \mid \text{regime A}]\).
- The **full-data model** is tuned to minimize loss over the mixture. So it tends toward a predictor that works well on *average* over all regimes—a **compromise**. That compromise is generally **not** the best predictor for regime A alone.
- So when we evaluate on data from A:
  - **Our method:** We load the checkpoint that was trained when we last saw regime A → a model optimized for \(P_A\) (or data dominated by A). So we use (approximately) the **specialist** for A.
  - **Full retrain:** Uses the **generalist** optimized for \(P_{\text{mix}}\). On data from A, its error can be **higher** than the specialist’s.

So we don’t beat “more data” in the abstract—we beat “one model forced to fit a mixture of distributions.” When the **evaluation distribution** is a single regime, the **specialist for that regime** can (and in our multi_regime experiments does) achieve better accuracy than the **generalist** trained on the full mixture.

In [ ]:
# Numerical proof: full-data (mixture) predictor has strictly higher MSE on current regime
# Setup: Regimes A and B, means μ_A, μ_B, same variance σ². Test batch is from A.
# Specialist (ours): predicts μ_A → MSE = E[(Y - μ_A)²] = σ².
# Full retrain: one model trained on A∪B → optimal for mixture is μ_mix = (n_A μ_A + n_B μ_B)/N.
# On test data Y ~ N(μ_A, σ²): MSE_full = E[(Y - μ_mix)²] = σ² + (μ_A - μ_mix)² ≥ σ², with equality only if μ_mix = μ_A.

import numpy as np
np.random.seed(42)
mu_a, mu_b, sigma = 0.0, 5.0, 1.0
n_a, n_b = 40, 40  # equal history from both regimes
mu_mix = (n_a * mu_a + n_b * mu_b) / (n_a + n_b)  # = 2.5

# Test batch from regime A
y_test = np.random.normal(mu_a, sigma, 200)
mse_specialist = np.mean((y_test - mu_a) ** 2)
mse_full_retrain = np.mean((y_test - mu_mix) ** 2)

print("Test data: from Regime A (mean=0, σ²=1).")
print(f"  Specialist (ours): predict μ_A = {mu_a}  → MSE = {mse_specialist:.4f} (= σ² in theory)")
print(f"  Full retrain:      predict μ_mix = {mu_mix} → MSE = {mse_full_retrain:.4f} (= σ² + (μ_A - μ_mix)²)")
print(f"  Extra error from full retrain: (μ_A - μ_mix)² = {(mu_a - mu_mix)**2:.4f}")
print("\n→ On data from one regime, the mixture-optimal predictor has strictly higher MSE than the regime specialist.")

### How this matches our multi_regime experiments

- **Multi_regime synthetic data** (see `benchmarks/data_loaders/synthetic_generator.py`): 4 regimes with different `base`, `trend`, `seasonal_amp`, `noise_std`. Data alternates across these regimes; the same regime can **recur** later.
- **Full retrain:** At each step, the baseline fits one GRU on *all* points so far (e.g. mix of regimes 1–3). That model is a compromise over those regimes.
- **Ours:** When the new batch’s distribution matches a stored regime (e.g. regime 2), we load the checkpoint trained on regime-2 data. So we use a **regime-2 specialist** for the current batch.
- **Result:** On the 7 multi_regime datasets, regime-aware reuse **wins vs both** full retrain and TTA, because the current batch is from a recurring regime and we deploy the right specialist; full retrain keeps using the compromise model; TTA only adapts on the small new batch and can underfit or drift.
- **Paper takeaway:** *We can beat full-data training when the data stream has recurring regimes and we evaluate on the current regime: the full-data model is optimal for the mixture of all past regimes, while we use a model optimized for the current regime’s distribution.*

In [ ]:
# Datasets where regime wins vs BOTH baseline and TTA (strong wins)
strong_wins = df[(df["vs_baseline"] == "regime_wins") & (df["vs_tta"] == "regime_wins")]
print("Regime wins vs BOTH baseline and TTA:")
print(strong_wins[["dataset", "scenario", "baseline_mape", "tta_mape", "regime_mape"]].to_string())

# Datasets where regime loses vs both
strong_losses = df[(df["vs_baseline"] == "baseline_wins") & (df["vs_tta"] == "tta_wins")]
print("\nRegime loses vs BOTH:")
print(strong_losses[["dataset", "scenario", "baseline_mape", "tta_mape", "regime_mape"]].to_string())

## 4. Hypotheses: why we win in some cases

**When regime wins:**
- **Recurring regimes (multi_regime):** Data switches between a few regimes. We save a checkpoint per regime; when the same regime reappears, we load that checkpoint. Full retrain mixes all history → can dilute regime-specific signal. TTA only adapts on the new batch → may underfit or drift. We use a model *trained specifically on that regime* → better fit.
- **Stable / recurring patterns:** If the *distribution* of the new batch is similar to a past regime (mean, std, skew, etc.), the matched checkpoint is the right specialist.

**When regime loses:**
- **Novel shock (covid_shock):** One-off shift; no past regime matches. We may incorrectly match to a pre-shock regime and use the wrong model. Full retrain sees the new data; TTA adapts to it. We reuse a wrong checkpoint → worse.
- **High volatility / no clear regimes:** Similarity is noisy; wrong matches or many "no match" retrains. TTA’s in-place adaptation can track recent data better.
- **Stocks (some):** Fast-changing, regime signatures may overlap; wrong match hurts more than retrain or TTA.

**Numerical intuition:** Suppose two regimes A and B. Full retrain trains on A∪B → one model for both. TTA starts from current model and updates on latest batch only. Ours: when batch is from A again, we load the A-checkpoint (trained on A). If the data is actually from A, our predictor is optimal for A; full retrain’s model is compromised by B; TTA may have drifted. So we win when *regime reoccurs and we match correctly*.

## 5. Simple numerical illustration: two-regime setting

Idea: two regimes (different means). When the current batch is from regime A again:
- **Ours:** Load checkpoint trained on A → error ~ variance of A.
- **Full retrain:** Model trained on A∪B → can have higher error if B pulls the estimate away from A.
- **TTA:** Model was maybe last updated on B; a few steps on small batch from A may underfit.

We can’t run the real GRU here, but we can compare *optimal* predictors in a toy Gaussian setting.

In [ ]:
# Toy: Regime A = N(0,1), Regime B = N(5,1). Next batch is from A (mean=0).
# Optimal predictor for A is mean 0; for A∪B is mean 2.5.
# If we predict with mean 0 (ours, correct regime): MSE = 1.
# If we predict with mean 2.5 (full retrain mix): MSE = E[(y-2.5)^2] for y~N(0,1) = 1 + 6.25 = 7.25.
# So in this toy, "ours" (correct regime) has much lower error than full retrain when batch is from A.

import numpy as np

np.random.seed(42)
n = 500
mu_a, mu_b, sigma = 0.0, 5.0, 1.0
y_a = np.random.normal(mu_a, sigma, n)
y_b = np.random.normal(mu_b, sigma, n)

# Next batch from regime A
y_new = np.random.normal(mu_a, sigma, 50)

pred_ours = mu_a           # we loaded A checkpoint
pred_full_retrain = (mu_a + mu_b) / 2  # model trained on A union B
pred_tta_bias = 4.0       # TTA drifted toward B, then a few steps on small A batch → still biased

mse_ours = np.mean((y_new - pred_ours) ** 2)
mse_full = np.mean((y_new - pred_full_retrain) ** 2)
mse_tta = np.mean((y_new - pred_tta_bias) ** 2)

print("Toy: next batch from Regime A (mean=0). Predictors: ours=0, full_retrain=2.5, TTA=4 (biased).")
print(f"MSE (ours, correct regime):     {mse_ours:.4f}")
print(f"MSE (full retrain on A∪B):      {mse_full:.4f}")
print(f"MSE (TTA biased toward B):      {mse_tta:.4f}")
print("\nConclusion: When regime reoccurs and we match correctly, we win by using the right specialist.")

## 6. When we lose: wrong match

If we *wrongly* match to regime B when the batch is from A:
- We predict with mean 5 (B’s mean) for data from A (mean 0) → large error.
- Full retrain and TTA (after adapting) use information from the current batch → can do better.

That explains covid_shock and some volatile stocks: novel or shifting distribution, but similarity might match to an old regime → wrong checkpoint.

In [ ]:
# Wrong match: batch is from A, but we load B checkpoint (pred = 5)
pred_wrong_match = mu_b
mse_wrong = np.mean((y_new - pred_wrong_match) ** 2)
print(f"MSE (wrong match: use B for A data): {mse_wrong:.4f}")
print("So wrong match can be much worse than full retrain or TTA.")

## 7. Summary table for the paper

Summarise: (1) scenario types where regime wins vs baseline and vs TTA; (2) one-sentence reason (recurring regime → right checkpoint; novel shock → wrong match; etc.).

In [ ]:
summary = df.groupby("scenario").agg(
    count=("dataset", "count"),
    regime_wins_vs_baseline=("vs_baseline", lambda s: (s == "regime_wins").sum()),
    regime_wins_vs_tta=("vs_tta", lambda s: (s == "regime_wins").sum()),
    avg_regime_mape=("regime_mape", "mean"),
    avg_baseline_mape=("baseline_mape", "mean"),
    avg_tta_mape=("tta_mape", "mean"),
).round(2)
summary

In [ ]:
# Export key tables for paper (same dir as three_way_results.json)
out_dir = results_path.parent
strong_wins.to_csv(out_dir / "regime_strong_wins.csv", index=False)
strong_losses.to_csv(out_dir / "regime_strong_losses.csv", index=False)
summary.to_csv(out_dir / "why_regime_wins_summary_by_scenario.csv")
print("Exported:", str(out_dir), "| regime_strong_wins.csv, regime_strong_losses.csv, why_regime_wins_summary_by_scenario.csv")